In [1]:
import os

print("INPUT ROOT:")
print(os.listdir("/kaggle/input"))

print("\nDATASETS:")
print(os.listdir("/kaggle/input/datasets"))

print("\nSAUTKIN DATASETS:")
print(os.listdir("/kaggle/input/datasets/sautkin"))

INPUT ROOT:
['datasets']

DATASETS:
['sautkin']

SAUTKIN DATASETS:
['imagenet1kvalid', 'imagenet1k2', 'imagenet1k0', 'imagenet1k3', 'imagenet1k1']


## Clean the Working Directory

In [2]:
import shutil
import os

for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)

    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

In [ ]:
!rm -rf LSNET-advanced
!git clone -b proposal_7 https://github.com/Param45/LSNET-advanced.git
%cd LSNET-advanced
!ls

Cloning into 'LSNET-advanced'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 254 (delta 12), reused 14 (delta 6), pack-reused 228 (from 1)
Receiving objects: 100% (254/254), 41.51 MiB | 39.84 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/kaggle/working/LSNET-advanced
data		figures		  logs	     README.md		   segmentation
detection	flops.py	  losses.py  README_robustness.md  speed.py
engine.py	kaggle_config.py  main.py    requirements.txt	   train.sh
eval_robust.sh	kaggle_run.py	  model      robust.py		   utils.py
eval.sh		KAGGLE_SETUP.md   pretrain   robust_utils.py


In [4]:
import torch
print(torch.__version__)

2.10.0+cu128


In [5]:
!pip install -q timm fvcore wandb
!pip install timm==0.5.4 einops==0.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but yo

In [6]:
from kaggle_config import IMAGENET_PATH

print(IMAGENET_PATH)

/kaggle/input/datasets/sautkin


In [7]:
import os

for ds in [
    "imagenet1k0",
    "imagenet1k1",
    "imagenet1k2",
    "imagenet1k3",
    "imagenet1kvalid"
]:
    path = f"/kaggle/input/datasets/sautkin/{ds}"
    print(ds, len(os.listdir(path)))

imagenet1k0 500
imagenet1k1 500
imagenet1k2 500
imagenet1k3 500
imagenet1kvalid 1000


## Evaluation: Prediction Visualisation & Comprehensive Metrics

The cell below loads the **pretrained LSNet-T weights** (`pretrain/lsnet_t.pth`) and runs inference-only:
1. Builds the validation DataLoader (40 000 images, 1 000 classes).
2. Calls `visualize_predictions()` — displays **4 randomly sampled validation images** in a **2×2 matplotlib grid**.  
   Each subplot title shows the **Ground-Truth label**, **Predicted label**, and **Top-1 Softmax confidence (%)**.
3. Calls `evaluate_comprehensive()` — runs a full evaluation pass and prints **all relevant metrics** below the figure:
   Top-1 / Top-5 accuracy, cross-entropy loss, macro-averaged class accuracy, Precision / Recall / F1 (macro, micro, weighted), inference latency, and per-class accuracy statistics.

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation-only: Visualisation & Comprehensive Metrics
# Uses the pretrained LSNet-T weights; NO training is performed.
# ---------------------------------------------------------------------------

import os, sys, random
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend safe for Kaggle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from types import SimpleNamespace

# ── 1. Ensure project root is on sys.path ─────────────────────────────────
PROJECT_ROOT = '/kaggle/working/LSNET-advanced'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from data.datasets import build_dataset
from model import build  # registers lsnet_t (and other variants) with timm
from timm.models import create_model
from timm.utils import accuracy as timm_accuracy
import utils as lsnet_utils

# ── 2. Checkpoint — pretrained weights only ───────────────────────────────
CHECKPOINT_PATH = '/kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth'
print(f'Checkpoint : {CHECKPOINT_PATH}')

# ── 3. Build args (mirrors the project's default eval configuration) ──────
args = SimpleNamespace(
    data_set='IMNET',
    data_path='/kaggle/input/datasets/sautkin',
    input_size=224,
    color_jitter=0.4,
    aa='rand-m9-mstd0.5-inc1',
    train_interpolation='bicubic',
    reprob=0.25,
    remode='pixel',
    recount=1,
    # finetune='...' triggers Resize-only eval transform (no CenterCrop)
    finetune=CHECKPOINT_PATH,
    inat_category='name',
    nb_classes=1000,
    batch_size=512,
    num_workers=4,
    pin_mem=True,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {DEVICE}')

# ── 4. Validation DataLoader ──────────────────────────────────────────────
dataset_val, n_classes = build_dataset(is_train=False, args=args)
sampler_val = torch.utils.data.SequentialSampler(dataset_val)
data_loader_val = torch.utils.data.DataLoader(
    dataset_val,
    sampler=sampler_val,
    batch_size=int(2.0 * args.batch_size),
    num_workers=args.num_workers,
    pin_memory=args.pin_mem,
    drop_last=False,
    persistent_workers=False,
)
print(f'Val samples: {len(dataset_val):,} | Classes: {n_classes}')

# ── 5. Build & load the model ─────────────────────────────────────────────
model = create_model('lsnet_t', num_classes=n_classes)

ckpt_raw = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
if   'model'      in ckpt_raw: state_dict = ckpt_raw['model']
elif 'state_dict' in ckpt_raw: state_dict = ckpt_raw['state_dict']
else:                          state_dict = ckpt_raw

model_sd = model.state_dict()
filtered = {
    k: v for k, v in state_dict.items()
    if k in model_sd and model_sd[k].shape == v.shape
}
missing, unexpected = model.load_state_dict(filtered, strict=False)
if missing:    print(f'  Missing keys    ({len(missing)}): {missing[:5]}')
if unexpected: print(f'  Unexpected keys ({len(unexpected)}): {unexpected[:5]}')

model.to(DEVICE).eval()
lsnet_utils.replace_batchnorm(model)   # merge Conv-BN for inference
print('Model loaded and ready for inference.')

# ── 6. ImageNet class-name mapping ────────────────────────────────────────
# Priority order:
#   a) class_number_to_label.txt (class number -> human-readable label)
#   b) imagenet_class_index.json (synset -> [wnid, human_label])
#   c) dataset.class_to_idx      (synset_folder -> int index)
IMAGENET_CLASSES: dict = {}   # int index -> human-readable name

# Try loading from class_number_to_label.txt
import glob
_txt_candidates = [
    '/kaggle/working/LSNET-advanced/class_number_to_label.txt',
    '/kaggle/working/class_number_to_label.txt',
    '/kaggle/input/class-number-to-label/class_number_to_label.txt',
]
try:
    # Add any recursively found class_number_to_label.txt files under /kaggle
    _txt_candidates.extend(glob.glob('/kaggle/input/**/class_number_to_label.txt', recursive=True))
except Exception:
    pass

for _p in _txt_candidates:
    if os.path.exists(_p):
        try:
            with open(_p, 'r', encoding='utf-8') as _f:
                for _line in _f:
                    _line = _line.strip()
                    if not _line:
                        continue
                    _parts = _line.split(',', 1)
                    if len(_parts) == 2 and _parts[0].strip().isdigit():
                        _idx = int(_parts[0].strip())
                        _label = _parts[1].strip()
                        IMAGENET_CLASSES[_idx] = _label
            if IMAGENET_CLASSES:
                print(f'Loaded class names from {_p}')
                break
        except Exception as _e:
            print(f'Error reading class labels from {_p}: {_e}')

if not IMAGENET_CLASSES:
    try:
        import json as _json
        _candidates = [
            '/kaggle/working/LSNET-advanced/data/imagenet_class_index.json',
            os.path.join(os.path.dirname(torch.__file__), 'hub', 'imagenet_classes.json'),
        ]
        for _p in _candidates:
            if os.path.exists(_p):
                with open(_p) as _f:
                    _raw = _json.load(_f)
                IMAGENET_CLASSES = {int(k): v[1] for k, v in _raw.items()}
                print(f'Loaded class names from {_p}')
                break
    except Exception as _e:
        print(f'Could not load class-index JSON: {_e}')

if not IMAGENET_CLASSES:
    if hasattr(dataset_val, 'class_to_idx'):
        IMAGENET_CLASSES = {v: k for k, v in dataset_val.class_to_idx.items()}
        print('Using dataset synset folder names as class labels.')

def class_name(idx: int) -> str:
    """Return a human-readable label for class `idx` (max 30 chars)."""
    label = IMAGENET_CLASSES.get(int(idx), None)
    if label is None:
        return f'class_{idx}'
    return label[:30]


# ============================================================
# visualize_predictions()
# ============================================================
def visualize_predictions(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    n: int = 4,
    seed: int = 42,
    grid_id: int = 1,
) -> None:
    """
    Sample `n` images from the existing validation DataLoader, run
    inference, and display a 2x2 matplotlib grid.

    Each subplot shows:
      - the image (de-normalised)
      - GT class name  (human-readable if available, synset ID otherwise)
      - Predicted class name
      - Top-1 softmax confidence (%)

    No new DataLoader is created — a Subset of data_loader.dataset is used.
    GPU-to-CPU conversion is done via .detach().cpu().numpy().

    Parameters
    ----------
    grid_id  : used only for the output filename and title disambiguation.
    """
    assert n == 4, 'Designed for exactly 4 images (2x2 grid).'

    rng     = random.Random(seed)
    total   = len(data_loader.dataset)
    indices = sorted(rng.sample(range(total), n))

    # Subset of the EXISTING dataset — same transforms, no new loader
    subset        = torch.utils.data.Subset(data_loader.dataset, indices)
    subset_loader = torch.utils.data.DataLoader(
        subset, batch_size=n, num_workers=0, shuffle=False
    )

    model.eval()
    with torch.no_grad():
        imgs_batch, labels_batch = next(iter(subset_loader))

    # ── Inference ────────────────────────────────────────────────────────
    inp = imgs_batch.to(device)
    if inp.dtype == torch.uint8:   # GPU-aug pipeline emits uint8
        from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
        from torchvision.transforms.functional import normalize
        inp = inp.float().div(255.0)
        inp = normalize(inp, IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)

    if device.type == 'cuda':
        with torch.amp.autocast(enabled=True, dtype=torch.float16, device_type='cuda'):
            logits = model(inp)
    else:
        with torch.no_grad():
            logits = model(inp)

    probs            = F.softmax(logits.float(), dim=-1)
    confs, pred_idxs = probs.max(dim=-1)   # top-1

    # ── GPU → CPU (detach before numpy) ──────────────────────────────────
    imgs_np   = imgs_batch.detach().cpu().numpy()     # [4, C, H, W]
    labels_np = labels_batch.detach().cpu().numpy()   # [4]
    confs_np  = confs.detach().cpu().float().numpy()  # [4]
    preds_np  = pred_idxs.detach().cpu().numpy()      # [4]

    # ── De-normalise for display ──────────────────────────────────────────
    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def to_display(t_chw):
        if t_chw.dtype == np.uint8:
            return t_chw.transpose(1, 2, 0)
        img = t_chw.transpose(1, 2, 0)   # HWC float32
        img = img * STD + MEAN
        return np.clip(img, 0.0, 1.0)

    # ── 2x2 plot ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(
        f'LSNet-T (Pretrained) — Validation Predictions  [Grid {grid_id}/4]',
        fontsize=13, fontweight='bold', y=1.01
    )

    for i, ax in enumerate(axes.flat):
        gt_idx  = int(labels_np[i])
        pd_idx  = int(preds_np[i])
        conf    = float(confs_np[i]) * 100.0
        correct = (gt_idx == pd_idx)

        ax.imshow(to_display(imgs_np[i]))
        ax.axis('off')

        border_col = '#2ecc71' if correct else '#e74c3c'
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col)
            spine.set_linewidth(3)
            spine.set_visible(True)

        ax.set_title(
            f'GT:   {class_name(gt_idx)}\n'
            f'Pred: {class_name(pd_idx)}\n'
            f'Conf: {conf:.1f}%',
            fontsize=10,
            color='#155724' if correct else '#721c24',
            pad=6,
        )

    correct_patch = mpatches.Patch(color='#2ecc71', label='Correct prediction')
    wrong_patch   = mpatches.Patch(color='#e74c3c', label='Incorrect prediction')
    fig.legend(handles=[correct_patch, wrong_patch],
               loc='lower center', ncol=2, fontsize=11,
               framealpha=0.85, bbox_to_anchor=(0.5, -0.03))

    plt.tight_layout()
    out_path = f'/kaggle/working/prediction_grid_{grid_id}.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  Grid {grid_id} saved -> {out_path}')


# ============================================================
# evaluate_comprehensive()
# ============================================================
def evaluate_comprehensive(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
) -> dict:
    """
    Full evaluation pass.  Returns a dict containing:
      top1, top5, mean_loss
      macro_cls_acc
      prec/rec/f1  (macro, micro, weighted)  [requires scikit-learn]
      avg_batch_latency_ms, per_sample_latency_ms
      per_cls_acc   (numpy array, one value per class)
      all_preds     (numpy int array, shape [N])   <- needed for confusion matrix
      all_labels    (numpy int array, shape [N])
    """
    import time as _t
    criterion = torch.nn.CrossEntropyLoss()

    all_preds_list, all_labels_list = [], []
    total_loss = total_top1 = total_top5 = 0.0
    n_samples  = 0
    total_time_ms = 0.0

    model.eval()
    print('\nRunning full evaluation pass ...')
    with torch.no_grad():
        for bi, (imgs, labels) in enumerate(data_loader):
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if imgs.dtype == torch.uint8:
                from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
                from torchvision.transforms.functional import normalize
                imgs = imgs.float().div(255.0)
                imgs = normalize(imgs, IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)

            t0 = _t.perf_counter()
            with torch.amp.autocast(
                enabled=(device.type == 'cuda'),
                dtype=torch.float16,
                device_type=device.type
            ):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = _t.perf_counter()

            bs = imgs.size(0)
            total_time_ms += (t1 - t0) * 1000.0
            total_loss    += loss.item() * bs

            a1, a5 = timm_accuracy(logits, labels, topk=(1, 5))
            total_top1 += a1.item() * bs / 100.0
            total_top5 += a5.item() * bs / 100.0

            all_preds_list.append(logits.argmax(dim=-1).detach().cpu())
            all_labels_list.append(labels.detach().cpu())
            n_samples += bs

            if (bi + 1) % 5 == 0:
                print(f'  [{bi+1:3d}/{len(data_loader)}]  '
                      f'running top-1: {100.0*total_top1/n_samples:.2f}%')

    all_preds  = torch.cat(all_preds_list).numpy()
    all_labels = torch.cat(all_labels_list).numpy()

    top1      = 100.0 * total_top1 / n_samples
    top5      = 100.0 * total_top5 / n_samples
    mean_loss = total_loss / n_samples
    avg_lat   = total_time_ms / len(data_loader)
    per_sam   = total_time_ms / n_samples

    # Per-class accuracy (macro)
    n_cls = int(all_labels.max()) + 1
    per_cls_correct = np.zeros(n_cls, dtype=np.int64)
    per_cls_total   = np.zeros(n_cls, dtype=np.int64)
    for gt, pd in zip(all_labels, all_preds):
        per_cls_total[gt]   += 1
        per_cls_correct[gt] += int(gt == pd)
    per_cls_acc   = per_cls_correct / np.maximum(per_cls_total, 1)
    macro_cls_acc = per_cls_acc.mean() * 100.0

    try:
        from sklearn.metrics import precision_score, recall_score, f1_score
        kw = dict(zero_division=0)
        pm = precision_score(all_labels, all_preds, average='macro',    **kw) * 100
        pi = precision_score(all_labels, all_preds, average='micro',    **kw) * 100
        pw = precision_score(all_labels, all_preds, average='weighted', **kw) * 100
        rm = recall_score(   all_labels, all_preds, average='macro',    **kw) * 100
        ri = recall_score(   all_labels, all_preds, average='micro',    **kw) * 100
        rw = recall_score(   all_labels, all_preds, average='weighted', **kw) * 100
        fm = f1_score(       all_labels, all_preds, average='macro',    **kw) * 100
        fi = f1_score(       all_labels, all_preds, average='micro',    **kw) * 100
        fw = f1_score(       all_labels, all_preds, average='weighted', **kw) * 100
        has_sklearn = True
    except ImportError:
        pm = pi = pw = rm = ri = rw = fm = fi = fw = None
        has_sklearn = False

    return dict(
        top1=top1, top5=top5, mean_loss=mean_loss,
        macro_cls_acc=macro_cls_acc,
        prec_macro=pm,   prec_micro=pi,   prec_weighted=pw,
        rec_macro=rm,    rec_micro=ri,    rec_weighted=rw,
        f1_macro=fm,     f1_micro=fi,     f1_weighted=fw,
        has_sklearn=has_sklearn,
        avg_batch_latency_ms=avg_lat,
        per_sample_latency_ms=per_sam,
        n_samples=n_samples,
        per_cls_acc=per_cls_acc,
        all_preds=all_preds,
        all_labels=all_labels,
    )


# ============================================================
# plot_confusion_matrix()
# ============================================================
def plot_confusion_matrix(
    all_labels: np.ndarray,
    all_preds:  np.ndarray,
    top_n: int = 30,
) -> None:
    """
    Plot a focused confusion matrix.

    Because a full 1000x1000 matrix is unreadable, we:
      1. Compute the full confusion matrix (saved as .npz for later use).
      2. Find the top_n classes with the highest off-diagonal error counts
         (i.e. the most mis-classified classes).
      3. Plot a top_n x top_n sub-matrix for those classes.

    The heatmap uses a log-scale colorbar so rare errors are visible
    alongside frequent ones.
    """
    try:
        from sklearn.metrics import confusion_matrix as sk_cm
    except ImportError:
        print('scikit-learn not available — skipping confusion matrix.')
        return

    n_cls = int(max(all_labels.max(), all_preds.max())) + 1
    print(f'\nComputing {n_cls}x{n_cls} confusion matrix ...')
    cm = sk_cm(all_labels, all_preds, labels=list(range(n_cls)))

    # Save full matrix compressed
    npz_path = '/kaggle/working/confusion_matrix_full.npz'
    np.savez_compressed(npz_path, cm=cm)
    print(f'Full {n_cls}x{n_cls} confusion matrix saved -> {npz_path}')

    # ── Select top-N most-confused classes ───────────────────────────────
    # "off-diagonal errors for class i" = row_sum(i) - cm[i,i]
    row_errors = cm.sum(axis=1) - np.diag(cm)   # errors per true class
    top_idx    = np.argsort(row_errors)[-top_n:][::-1]   # descending
    top_idx    = np.sort(top_idx)                         # sort for readability

    cm_sub  = cm[np.ix_(top_idx, top_idx)].astype(float)
    labels_ = [class_name(int(i)) for i in top_idx]

    # ── Plot ──────────────────────────────────────────────────────────────
    fig_size = max(14, top_n * 0.55)
    fig, ax  = plt.subplots(figsize=(fig_size, fig_size * 0.9))

    # Use log1p scale so low values are visible
    import matplotlib.colors as mcolors
    im = ax.imshow(
        np.log1p(cm_sub),
        cmap='Blues',
        aspect='auto',
    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('log(count + 1)', fontsize=11)

    ax.set_xticks(range(len(top_idx)))
    ax.set_yticks(range(len(top_idx)))
    ax.set_xticklabels(labels_, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labels_, fontsize=8)

    ax.set_xlabel('Predicted class', fontsize=12)
    ax.set_ylabel('True class',      fontsize=12)
    ax.set_title(
        f'Confusion Matrix — Top-{top_n} Most-Confused Classes\n'
        f'(diagonal = correct; off-diagonal = errors; color scale: log)',
        fontsize=13, fontweight='bold', pad=12,
    )

    # Annotate cells with raw counts (only if matrix is small enough)
    if top_n <= 20:
        for r in range(len(top_idx)):
            for c in range(len(top_idx)):
                val = int(cm_sub[r, c])
                if val > 0:
                    ax.text(c, r, str(val),
                            ha='center', va='center',
                            fontsize=7,
                            color='white' if cm_sub[r, c] > cm_sub.max() * 0.5 else 'black')

    plt.tight_layout()
    cm_path = f'/kaggle/working/confusion_matrix_top{top_n}.png'
    plt.savefig(cm_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Confusion matrix (top {top_n}) saved -> {cm_path}')

    # ── Print top-10 most-confused class pairs ────────────────────────────
    SEP = '-' * 62
    print(f'\n{SEP}')
    print(f'  Top-10 most-confused class pairs (true -> predicted)')
    print(SEP)
    # collect off-diagonal entries
    pairs = []
    for r in range(n_cls):
        for c in range(n_cls):
            if r != c and cm[r, c] > 0:
                pairs.append((cm[r, c], r, c))
    pairs.sort(reverse=True)
    for count, r, c in pairs[:10]:
        print(f'  {class_name(r):30s}  ->  {class_name(c):30s}  : {count} samples')
    print(SEP)


def print_metrics(m: dict) -> None:
    """Pretty-print the metrics dict returned by evaluate_comprehensive()."""
    SEP = '=' * 62
    print()
    print(SEP)
    print('  LSNet-T (Pretrained)  --  Comprehensive Evaluation Metrics')
    print(SEP)
    print(f'  Validation samples               : {m["n_samples"]:,}')
    print(f'  Cross-Entropy Loss (mean)        : {m["mean_loss"]:.4f}')
    print()
    print('  -- Accuracy --------------------------------------------------')
    print(f'  Top-1 Accuracy                   : {m["top1"]:.3f} %')
    print(f'  Top-5 Accuracy                   : {m["top5"]:.3f} %')
    print(f'  Macro-Averaged Class Accuracy    : {m["macro_cls_acc"]:.3f} %')
    print()
    if m['has_sklearn']:
        print('  -- Precision -------------------------------------------------')
        print(f'  Precision  (macro)               : {m["prec_macro"]:.3f} %')
        print(f'  Precision  (micro)               : {m["prec_micro"]:.3f} %')
        print(f'  Precision  (weighted)            : {m["prec_weighted"]:.3f} %')
        print()
        print('  -- Recall ----------------------------------------------------')
        print(f'  Recall     (macro)               : {m["rec_macro"]:.3f} %')
        print(f'  Recall     (micro)               : {m["rec_micro"]:.3f} %')
        print(f'  Recall     (weighted)            : {m["rec_weighted"]:.3f} %')
        print()
        print('  -- F1 Score --------------------------------------------------')
        print(f'  F1         (macro)               : {m["f1_macro"]:.3f} %')
        print(f'  F1         (micro)               : {m["f1_micro"]:.3f} %')
        print(f'  F1         (weighted)            : {m["f1_weighted"]:.3f} %')
    else:
        print('  (Install scikit-learn for Precision / Recall / F1 metrics)')
    print()
    print('  -- Latency ---------------------------------------------------')
    print(f'  Avg batch latency                : {m["avg_batch_latency_ms"]:.1f} ms')
    print(f'  Per-sample latency               : {m["per_sample_latency_ms"]:.2f} ms')
    print()
    pca = m['per_cls_acc'] * 100.0
    print('  -- Per-Class Accuracy Statistics -----------------------------')
    print(f'  Min  class accuracy              : {pca.min():.2f} %')
    print(f'  Max  class accuracy              : {pca.max():.2f} %')
    print(f'  Median class accuracy            : {np.median(pca):.2f} %')
    print(f'  Std  class accuracy              : {pca.std():.2f} %')
    worst5 = np.argsort(pca)[:5]
    best5  = np.argsort(pca)[-5:][::-1]
    print()
    print('  -- Bottom-5 Classes (by accuracy) ---------------------------')
    for idx in worst5:
        print(f'    [{int(idx):4d}] {class_name(int(idx)):30s}  {pca[idx]:.1f}%')
    print()
    print('  -- Top-5 Classes (by accuracy) ------------------------------')
    for idx in best5:
        print(f'    [{int(idx):4d}] {class_name(int(idx)):30s}  {pca[idx]:.1f}%')
    print(SEP)


# ── Run everything ─────────────────────────────────────────────────────────

# STEP A: 4 separate 2x2 prediction grids (different random seeds)
SEEDS = [42, 123, 456, 789]
print('\n' + '=' * 62)
print('  STEP A: visualize_predictions()  (4 grids x 4 images)')
print('=' * 62)
for grid_id, seed in enumerate(SEEDS, start=1):
    print(f'\n  -- Grid {grid_id}/4  (seed={seed}) --')
    visualize_predictions(
        model, data_loader_val, DEVICE,
        n=4, seed=seed, grid_id=grid_id,
    )

# STEP B: Full evaluation pass
print('\n' + '=' * 62)
print('  STEP B: evaluate_comprehensive()  (full val pass)')
print('=' * 62)
metrics = evaluate_comprehensive(model, data_loader_val, DEVICE)
print_metrics(metrics)

# STEP C: Confusion matrix (top-30 most confused classes)
print('\n' + '=' * 62)
print('  STEP C: plot_confusion_matrix()  (top-30 most confused)')
print('=' * 62)
plot_confusion_matrix(metrics['all_labels'], metrics['all_preds'], top_n=30)
